# Protein-altering SNVs across a 20-gene clinical panel

**Question.** Across four unrelated 1000 Genomes individuals (HG00096, HG00129, HG00268, HG00348), how many high-confidence SNVs in the coding regions of a 20-gene clinical panel are protein-altering, and how do they split by consequence?

**High confidence** means site QUAL of at least 30, genotypes with depth below 10 set to missing, and no more than a quarter of genotypes missing at the site. **Protein-altering** means VEP's picked consequence includes missense, stop gained, stop lost, start lost, splice donor or splice acceptor.

**Inputs** come from `workflow/run_pipeline.sh`: the filtered SNV table, a genotype table and VEP tab-delimited annotation.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "..")).resolve()
VEP_FILE = PROJECT_DIR / "results" / "vep" / "annotated_snvs.tsv"
SNV_FILE = PROJECT_DIR / "results" / "variants" / "filtered_snvs.tsv"
GT_FILE = PROJECT_DIR / "results" / "variants" / "genotypes.tsv"
OUT_DIR = PROJECT_DIR / "results" / "summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROTEIN_ALTERING = ["stop_gained", "splice_acceptor_variant", "splice_donor_variant",
                    "stop_lost", "start_lost", "missense_variant"]
# Ensembl severity order, most severe first (subset relevant to SNVs)
SEVERITY = ["splice_acceptor_variant", "splice_donor_variant", "stop_gained", "stop_lost",
            "start_lost", "missense_variant", "splice_donor_5th_base_variant", "splice_region_variant",
            "splice_donor_region_variant", "splice_polypyrimidine_tract_variant",
            "incomplete_terminal_codon_variant", "start_retained_variant", "stop_retained_variant",
            "synonymous_variant", "coding_sequence_variant", "5_prime_UTR_variant", "3_prime_UTR_variant",
            "non_coding_transcript_exon_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_variant"]
RANK = {c: i for i, c in enumerate(SEVERITY)}

## 1. Load VEP annotation

In [ ]:
def read_vep_tab(path):
    """VEP --tab output: '##' metadata lines, then a header starting with '#Uploaded_variation'."""
    with open(path) as fh:
        for i, line in enumerate(fh):
            if line.startswith("#Uploaded_variation"):
                break
        else:
            raise ValueError(f"No '#Uploaded_variation' header in {path}")
    df = pd.read_csv(path, sep="\t", skiprows=i, dtype=str)
    return df.rename(columns={"#Uploaded_variation": "Uploaded_variation"})


def most_severe(consequence):
    terms = re.split(r"[,&]", consequence)
    return min(terms, key=lambda t: RANK.get(t, len(RANK)))


vep = read_vep_tab(VEP_FILE)
vep["most_severe"] = vep["Consequence"].map(most_severe)
vep["protein_altering"] = vep["most_severe"].isin(PROTEIN_ALTERING)

dupes = vep["Uploaded_variation"].duplicated().sum()
assert dupes == 0, f"{dupes} variants have more than one row; rerun VEP with --pick"

snvs = pd.read_csv(SNV_FILE, sep="\t", header=None, names=["CHROM", "POS", "ID", "REF", "ALT"])
assert set(vep["Uploaded_variation"]) <= set(snvs["ID"]), "VEP output contains IDs not in the filtered SNV set"
print(f"{len(snvs):,} high-confidence SNVs, {len(vep):,} annotated")

## 2. Answer: protein-altering count and breakdown

In [ ]:
pa = vep[vep["protein_altering"]]
breakdown = (pa["most_severe"].value_counts()
             .reindex(PROTEIN_ALTERING, fill_value=0)
             .rename("n_variants"))
breakdown.loc["protein_altering_total"] = int(breakdown.sum())
breakdown.loc["all_high_confidence_snvs"] = len(snvs)
breakdown.to_csv(OUT_DIR / "consequence_counts.csv", index_label="category")
breakdown.to_frame()

In [ ]:
by_gene = (pa.groupby(["SYMBOL", "most_severe"]).size().unstack(fill_value=0)
           .assign(total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False))
by_gene.to_csv(OUT_DIR / "protein_altering_by_gene.csv")
by_gene

## 3. Genotypes: who carries what

Counts of protein-altering sites per individual, split by heterozygous and homozygous alternate calls. Missing genotypes come from the depth mask.

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=str)
gt.columns = [re.sub(r"^#?\s*\[\d+\]", "", c).removesuffix(":GT") for c in gt.columns]
gt = gt.set_index("ID")
gt_pa = gt.loc[gt.index.intersection(pa["Uploaded_variation"])]

def classify(g):
    a = re.split(r"[/|]", g)
    if "." in a: return "missing"
    alt = sum(x != "0" for x in a)
    return {0: "hom_ref", 1: "het", 2: "hom_alt"}[alt]

carriers = gt_pa.apply(lambda col: col.map(classify).value_counts()).T.fillna(0).astype(int)
carriers = carriers.reindex(columns=["het", "hom_alt", "hom_ref", "missing"], fill_value=0)
carriers.to_csv(OUT_DIR / "carriers_per_sample.csv")
carriers

## 4. Quality check: transition / transversion ratio

Real SNV calls in coding sequence typically give a Ti/Tv near 3, and whole-genome calls near 2.0 to 2.1. Random sequencing errors push the ratio toward 0.5, so a value far below the expected range means the filters are letting noise through.

In [ ]:
transitions = {("A", "G"), ("G", "A"), ("C", "T"), ("T", "C")}
is_ti = [(r, a) in transitions for r, a in zip(snvs["REF"], snvs["ALT"])]
n_ti = int(np.sum(is_ti)); n_tv = len(snvs) - n_ti
titv = n_ti / n_tv if n_tv else float("nan")
print(f"Transitions {n_ti}, transversions {n_tv}, Ti/Tv = {titv:.2f}")
if titv < 2.0:
    print("NOTE: Ti/Tv below 2.0; inspect low-quality sites before relying on the counts.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
plot = breakdown.loc[PROTEIN_ALTERING][::-1]
ax.barh([c.replace("_variant", "").replace("_", " ") for c in plot.index], plot.values, color="#2c3e50")
for y, v in enumerate(plot.values):
    ax.text(v, y, f" {v}", va="center", fontsize=8)
ax.set(xlabel="SNVs", title="Protein-altering SNVs by consequence")
fig.tight_layout()
fig.savefig(OUT_DIR / "consequence_counts.png", dpi=200)

## 5. Assumptions and limits

The QUAL, depth and missingness thresholds are common defaults rather than tuned values; moving any of them changes the counts, so they sit at the top of the pipeline script where they are easy to vary. Restricting to canonical coding exons plus 50 bp means deep intronic and regulatory variants are out of scope by design. VEP consequences are predictions from transcript models: a missense call says the amino acid changes, not that the protein is damaged, and no pathogenicity scoring or ClinVar lookup is applied here. Four individuals from two European populations say nothing about allele frequency in any population. The alignments are NYGC's BWA-MEM alignments to GRCh38; reads were not realigned for this project.

In [ ]:
print("Python", sys.version.split()[0])
for mod in (np, pd, matplotlib):
    print(f"{mod.__name__:<12}{mod.__version__}")